### Use HPSv2 to evaluate human preference (Quality of Vendi Score)

* **Paper:** *Human Preference Score v2: A Solid Benchmark for Evaluating Human Preferences of Text-to-Image Synthesis*
* **Repository:** [tgxs002/HPSv2 – Image Comparison](https://github.com/tgxs002/HPSv2#image-comparison) -> MUST BE INSTALLED



In [ ]:
import re
from pathlib import Path
import pandas as pd
import hpsv2

# PRIVATE FUNCTION
def _extract_id_from_path(img_path: str) -> str:

    p = Path(img_path)

    for part in reversed(p.parts):
        m = re.search(r'round-\d+(?:-seed-\d+)?', part, flags=re.IGNORECASE)
        if m:
            return m.group(0)
    m = re.search(r'round-\d+(?:-seed-\d+)?', p.stem, flags=re.IGNORECASE)
    return m.group(0) if m else "unknown"


# PUBLIC FUNCTIONS AND ATTRIBUTES
def collect_prompts_and_pngs(parent_dir: str):
    parent = Path(parent_dir)
    prompt_dirs = sorted([p for p in parent.iterdir() if p.is_dir()])

    prompts = []
    png_2d = []

    for pdir in prompt_dirs:
        pretty = pdir.name.replace("-", " ").capitalize()
        prompts.append(pretty)

        imgs = [
            str(img.resolve())
            for r in sorted(pdir.glob("round-*"))
            if r.is_dir()
            for img in sorted(r.glob("*.png"))
        ]
        png_2d.append(imgs)

    assert len(prompts) == len(png_2d), "Mismatched prompts and generated image groups"
    return prompts, png_2d

def score_prompts_with_hpsv2(prompts, png_2d, hps_version: str = "v2.1") -> pd.DataFrame:
    assert len(prompts) == len(png_2d), "Mismatched prompts and generated image groups"

    rows = []
    for prompt_name, image_paths in zip(prompts, png_2d):
        for img_path in image_paths:
            try:
                score = hpsv2.score(img_path, prompt_name, hps_version=hps_version)
            except Exception:
                # If a particular image fails to score, capture it as None
                score = None
            rows.append({
                "prompt name": prompt_name,
                "ID": _extract_id_from_path(img_path),
                "Result": score,
            })

    df = pd.DataFrame(rows, columns=["prompt name", "ID", "Result"])
    return df


In [ ]:
images_paths = [
    "<art_image_folder_path>",
    "<art_muse_image_folder_path>",
    "<cuisine_image_folder_path>",
    "<landmarks_image_folder_path>"
]

prompt_list = []
png_list = []
for image_path in images_paths:
    prompts, pngs = collect_prompts_and_pngs(image_path)
    assert len(prompts) == len(pngs), "Mismatched prompts and generated image groups"
    
    prompt_list.append(prompts)
    png_list.append(pngs)
    
    
df_results = score_prompts_with_hpsv2(prompts, pngs, hps_version="v2.1")
print(df_results.head())

/home/ubuntu/.pyenv/versions/music_gen/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/.pyenv/versions/music_gen/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/ubuntu/Xin_Fan/CUBE-MT/evaluate/HPSv2/hpsv2/img_score.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


                          prompt name        ID    Result
0  Create an image showing local food  round-00  [0.2487]
1  Create an image showing local food  round-00  [0.2393]
2  Create an image showing local food  round-00  [0.2275]
3  Create an image showing local food  round-00  [0.2612]
4  Create an image showing local food  round-00   [0.235]
